In [ ]:
import pandas as pd

players_df = pd.read_csv("/Users/michaelliu/.cache/kagglehub/datasets/eoinamoore/historical-nba-data-and-player-box-scores/versions/515/Players.csv")
stats_df = pd.read_csv("/Users/michaelliu/.cache/kagglehub/datasets/eoinamoore/historical-nba-data-and-player-box-scores/versions/515/PlayerStatistics.csv")
stats_df['date'] = pd.to_datetime(stats_df['gameDate'])

In [ ]:
reg_season = stats_df[(stats_df['date'].between('1990-11-01', '2000-06-01')) & (stats_df['gameType'].isin(['Regular Season']))]
reg_season['numMinutes_num'] = pd.to_numeric(reg_season['numMinutes'], errors='coerce')
reg_season_played = reg_season[reg_season['numMinutes_num'] > 0]

In [ ]:
stat_cols = [
    'points', 'assists', 'reboundsTotal', 'blocks', 'steals', 'turnovers',
    'fieldGoalsAttempted', 'fieldGoalsMade', 'threePointersAttempted', 'threePointersMade',
    'freeThrowsAttempted', 'freeThrowsMade'
]

player_averages = reg_season_played.groupby('personId').agg({
    'firstName': 'first',
    'lastName': 'first',
    'gameId': 'count',
    **{col: 'mean' for col in stat_cols}
}).reset_index().rename(columns={'gameId': 'gamesPlayed'})

player_averages['personId'] = player_averages['personId'].astype(int)

# Merge player positions (guard, forward, center) from players_df
players_positions = players_df[['personId', 'guard', 'forward', 'center']]
player_averages = player_averages.merge(players_positions, on='personId', how='left')
player_averages[['guard', 'forward', 'center']] = player_averages[['guard', 'forward', 'center']].fillna(0).astype(int)

In [ ]:
# Calculate True Shooting percentage (TS%)
player_averages['trueShootingAttempts'] = player_averages['fieldGoalsAttempted'] + 0.44 * player_averages['freeThrowsAttempted']
player_averages['ts'] = player_averages['points'] / (2 * player_averages['trueShootingAttempts'])
player_averages['ts'] = player_averages['ts'].fillna(0.0)

# Calculate Fantasy Points
player_averages['fantasyPoints'] = (
    player_averages['points'] +
    1.5 * player_averages['reboundsTotal'] +
    1.5 * player_averages['assists'] +
    4 * player_averages['steals'] +
    4 * player_averages['blocks'] -
    2 * player_averages['turnovers']
)

In [ ]:
# Filter out players with no valid position
player_averages = player_averages[player_averages[['guard', 'forward', 'center']].any(axis=1)]

# Filter to players who played 20 or more games
player_averages = player_averages[player_averages['gamesPlayed'] >= 20]

# Sort players by fantasyPoints descending
player_averages = player_averages.sort_values(by='fantasyPoints', ascending=False)

In [ ]:
output_csv_path = 'player_averages_1990_00.csv'
player_averages.to_csv(output_csv_path, index=False)
print(f"Successfully calculated averages for {len(player_averages)} players and saved to {output_csv_path}")

In [ ]:
player_averages.head(50)